# Day 054 — Exercise 5: The ChatStore Facade

**What you'll build:** `ChatStore(db_path)` — the app's persistence facade. Methods: `start(title)`, `append(conversation_id, role, content)`, `history(conversation_id)`, `conversations()`.

**Why it matters:** The app shouldn't sprinkle sessions and commits through its handlers. `ChatStore` hides all of that behind four verbs, each opening a short session and committing atomically. The backend calls `store.append(...)` and `store.history(...)` — clean, and its data lives in a file that survives every restart.

## Provided: Setup + Models + all repository functions

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import tempfile
from datetime import datetime
from sqlalchemy import create_engine, ForeignKey, select, inspect as sa_inspect, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Conversation(Base):
    """One chat conversation. Has many Messages (one-to-many)."""
    __tablename__ = 'conversations'

    id:         Mapped[int]      = mapped_column(primary_key=True)
    title:      Mapped[str]      = mapped_column(default='New chat')
    created_at: Mapped[datetime] = mapped_column(default=datetime.utcnow)

    # relationship() is the ORM link (not a DB column). cascade deletes a
    # conversation's messages when the conversation is deleted.
    messages: Mapped[list['Message']] = relationship(
        back_populates='conversation', cascade='all, delete-orphan')


class Message(Base):
    """One message in a conversation. Belongs to one Conversation (many-to-one)."""
    __tablename__ = 'messages'

    id:              Mapped[int]      = mapped_column(primary_key=True)
    conversation_id: Mapped[int]      = mapped_column(ForeignKey('conversations.id'))
    role:            Mapped[str]      = mapped_column()
    content:         Mapped[str]      = mapped_column()
    created_at:      Mapped[datetime] = mapped_column(default=datetime.utcnow)

    conversation: Mapped['Conversation'] = relationship(back_populates='messages')


def memory_engine():
    """In-memory SQLite engine for tests. StaticPool makes every Session share the
    one in-memory database (see Day 44)."""
    return create_engine('sqlite:///:memory:',
                          connect_args={'check_same_thread': False},
                          poolclass=StaticPool)


def create_schema(engine) -> None:
    """Create every table registered on Base (CREATE TABLE IF NOT EXISTS)."""
    Base.metadata.create_all(engine)


def create_conversation(session, title: str = 'New chat') -> Conversation:
    """Insert a new conversation and flush so its auto id is assigned.
    The caller controls commit (unit-of-work pattern)."""
    conv = Conversation(title=title)
    session.add(conv)
    session.flush()
    return conv


def add_message(session, conversation_id: int, role: str, content: str) -> Message:
    """Append a message to a conversation via its foreign key, and flush to assign
    the id. The caller commits."""
    msg = Message(conversation_id=conversation_id, role=role, content=content)
    session.add(msg)
    session.flush()
    return msg


def get_messages(session, conversation_id: int) -> list:
    """Return the conversation's messages, in insertion order, as plain
    [{'role', 'content'}] dicts (safe to use after the session closes)."""
    stmt = (select(Message)
            .where(Message.conversation_id == conversation_id)
            .order_by(Message.id))
    rows = session.execute(stmt).scalars().all()
    return [{'role': m.role, 'content': m.content} for m in rows]


def list_conversations(session) -> list:
    """Return all conversations as [{'id', 'title', 'message_count'}] by id."""
    convs = session.execute(
        select(Conversation).order_by(Conversation.id)).scalars().all()
    return [{'id': c.id, 'title': c.title, 'message_count': len(c.messages)}
            for c in convs]


def make_engine(db_path: str):
    """File-backed SQLite engine — data SURVIVES a process restart. Creates the
    schema on first use (safe to call every startup)."""
    engine = create_engine(f'sqlite:///{db_path}')
    Base.metadata.create_all(engine)
    return engine


def column_exists(engine, table: str, column: str) -> bool:
    """True if `column` already exists on `table` (schema introspection)."""
    return column in [c['name'] for c in sa_inspect(engine).get_columns(table)]


def migrate_add_column(engine, table: str, column: str, sqltype: str = 'TEXT') -> bool:
    """A minimal, idempotent migration: add a column only if it is missing.
    Returns True if it added the column, False if it was already there.
    Safe to run on every startup — this is the essence of a migration."""
    if column_exists(engine, table, column):
        return False
    with engine.begin() as conn:
        conn.execute(text(f'ALTER TABLE {table} ADD COLUMN {column} {sqltype}'))
    return True

## Your Implementation

In [ ]:
class ChatStore:
    """Persistence facade: one file-backed DB, a short Session per method."""

    def __init__(self, db_path: str):
        # TODO: self.engine = make_engine(db_path)
        pass

    def start(self, title: str = 'New chat') -> int:
        # TODO: with Session(self.engine) as s:
        #     conv = create_conversation(s, title); s.commit(); return conv.id
        pass

    def append(self, conversation_id: int, role: str, content: str) -> int:
        # TODO: with Session(self.engine) as s:
        #     msg = add_message(s, conversation_id, role, content); s.commit(); return msg.id
        pass

    def history(self, conversation_id: int) -> list:
        # TODO: with Session(self.engine) as s:
        #     return get_messages(s, conversation_id)
        pass

    def conversations(self) -> list:
        # TODO: with Session(self.engine) as s:
        #     return list_conversations(s)
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    db_path = os.path.join(tempfile.mkdtemp(), 'store.db')

    # Check 1: ChatStore exposes the four methods
    try:
        assert 'ChatStore' in globals()
        for m in ('start', 'append', 'history', 'conversations'):
            assert hasattr(ChatStore, m), f'missing method: {m}'
        store = ChatStore(db_path)
        passed += 1; print('✅ Check 1: ChatStore has start/append/history/conversations')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: start returns an int id; append works
    try:
        cid = store.start('Session A')
        assert isinstance(cid, int), f'start must return an int id, got {type(cid).__name__}'
        store.append(cid, 'user', 'hi')
        store.append(cid, 'assistant', 'hello!')
        passed += 1; print(f'✅ Check 2: start -> id {cid}, append works')
    except Exception as e:
        print(f'❌ Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: history returns the messages in order
    try:
        h = store.history(cid)
        assert h == [{'role': 'user', 'content': 'hi'},
                     {'role': 'assistant', 'content': 'hello!'}], f'bad history: {h}'
        passed += 1; print('✅ Check 3: history() returns the saved turns')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: conversations() lists it with a message_count
    try:
        convs = store.conversations()
        row = [c for c in convs if c['id'] == cid][0]
        assert row['message_count'] == 2, f"expected 2, got {row['message_count']}"
        passed += 1; print('✅ Check 4: conversations() reports message_count')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: SAVED STATE survives a 'restart' (new ChatStore, same file)
    try:
        store2 = ChatStore(db_path)
        assert store2.history(cid) == [{'role': 'user', 'content': 'hi'},
                                       {'role': 'assistant', 'content': 'hello!'}], \
            'a fresh ChatStore on the same file must see the saved data'
        passed += 1; print('✅ Check 5: data persists across a restart 💾')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class ChatStore:
    """Persistence facade for the chat app. One file-backed database; each method
    opens a short-lived Session and commits atomically. The app calls these
    methods and never touches SQL — the same thin-shell pattern as the earlier
    days, now over a database."""

    def __init__(self, db_path: str):
        self.engine = make_engine(db_path)

    def start(self, title: str = 'New chat') -> int:
        with Session(self.engine) as s:
            conv = create_conversation(s, title)
            s.commit()
            return conv.id

    def append(self, conversation_id: int, role: str, content: str) -> int:
        with Session(self.engine) as s:
            msg = add_message(s, conversation_id, role, content)
            s.commit()
            return msg.id

    def history(self, conversation_id: int) -> list:
        with Session(self.engine) as s:
            return get_messages(s, conversation_id)

    def conversations(self) -> list:
        with Session(self.engine) as s:
            return list_conversations(s)
```

**Why this works:** `ChatStore` owns one file-backed engine and opens a short `Session` per method — the classic web pattern of a session per unit of work. Each method commits before returning, so state is durable immediately. Because the database is a file, a brand-new `ChatStore` pointed at the same path sees every prior conversation — which is exactly what Check 5 proves and what 'saved state' means. The backend holds one `ChatStore` and calls these four verbs.
</details>